In [1]:
# ============================================================
# DM4ML Assignment I - Step 5: Data Preparation and EDA
# Purpose:
#   - Load latest validated/raw Retailrocket + DummyJSON data
#   - Clean and preprocess interactions and catalog data
#   - Handle missing user-item interactions for recommender prep
#   - Encode categorical attributes
#   - Normalize numerical variables
#   - Generate EDA plots: histograms, popularity, sparsity heatmap
#   - Save prepared datasets ready for transformation
# ============================================================

# =========================
# 0) Imports
# =========================


import sys
!{sys.executable} -m pip uninstall -y matplotlib seaborn
!{sys.executable} -m pip install --no-cache-dir matplotlib seaborn
import sys
!{sys.executable} -m pip install --no-cache-dir scikit-learn scipy pyarrow


import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import csr_matrix, save_npz

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# =========================
# 1) Config
# =========================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports" / "preparation"
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"
METADATA_DIR = PROJECT_ROOT / "metadata"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

RETAILROCKET_RAW_BASE = DATA_ROOT / "raw" / "retailrocket"
DUMMYJSON_RAW_BASE = DATA_ROOT / "raw" / "dummyjson"

EVENT_WEIGHTS = {
    "view": 1.0,
    "addtocart": 3.0,
    "transaction": 5.0,
}

TOP_USERS_HEATMAP = 40
TOP_ITEMS_HEATMAP = 40

RUN_TS = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

# =========================
# 2) Helpers
# =========================
def find_latest_complete_batch(base_dir: Path, expected_files):
    if not base_dir.exists():
        return None

    candidates = []
    for load_date_dir in sorted(base_dir.glob("load_date=*")):
        for load_hour_dir in sorted(load_date_dir.glob("load_hour=*")):
            if all((load_hour_dir / f).exists() for f in expected_files):
                candidates.append(load_hour_dir)

    if not candidates:
        return None
    return sorted(candidates)[-1]


def latest_file(base_dir: Path, pattern: str):
    files = sorted(base_dir.rglob(pattern))
    return files[-1] if files else None


def save_plot(fig, filename):
    output = REPORTS_DIR / filename
    fig.tight_layout()
    fig.savefig(output, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return output


def summarize_df(df, name):
    return {
        "dataset": name,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "null_cells": int(df.isna().sum().sum()),
    }


def minmax_scale_columns(df, columns, suffix="_norm"):
    scaler = MinMaxScaler()
    existing = [c for c in columns if c in df.columns]
    if existing:
        scaled = scaler.fit_transform(df[existing].fillna(0))
        for i, col in enumerate(existing):
            df[f"{col}{suffix}"] = scaled[:, i]
    return df


# =========================
# 3) Locate latest source batches
# =========================
retailrocket_batch = find_latest_complete_batch(
    RETAILROCKET_RAW_BASE,
    ["events.csv", "category_tree.csv", "item_properties_part1.csv", "item_properties_part2.csv"],
)

dummyjson_batch = find_latest_complete_batch(
    DUMMYJSON_RAW_BASE,
    ["products_raw.json", "categories_raw.json"],
)

if retailrocket_batch is None:
    raise FileNotFoundError("Retailrocket raw batch not found.")
if dummyjson_batch is None:
    raise FileNotFoundError("DummyJSON raw batch not found.")

print("Retailrocket batch:", retailrocket_batch)
print("DummyJSON batch:", dummyjson_batch)

# =========================
# 4) Load source data
# =========================
events = pd.read_csv(retailrocket_batch / "events.csv")
category_tree = pd.read_csv(retailrocket_batch / "category_tree.csv")
item_props_1 = pd.read_csv(retailrocket_batch / "item_properties_part1.csv")
item_props_2 = pd.read_csv(retailrocket_batch / "item_properties_part2.csv")
item_properties = pd.concat([item_props_1, item_props_2], ignore_index=True)

with open(dummyjson_batch / "products_raw.json", "r", encoding="utf-8") as f:
    products_payload = json.load(f)

with open(dummyjson_batch / "categories_raw.json", "r", encoding="utf-8") as f:
    categories_payload = json.load(f)

products = pd.json_normalize(products_payload.get("products", []), sep="_")

if isinstance(categories_payload, list):
    if len(categories_payload) > 0 and isinstance(categories_payload[0], dict):
        categories = pd.json_normalize(categories_payload, sep="_")
    else:
        categories = pd.DataFrame({"category": categories_payload})
else:
    categories = pd.DataFrame(columns=["category"])

print("Loaded datasets:")
print("events:", events.shape)
print("category_tree:", category_tree.shape)
print("item_properties:", item_properties.shape)
print("products:", products.shape)
print("categories:", categories.shape)

# =========================
# 5) Clean Retailrocket interactions
# =========================
events = events.copy()

required_event_cols = ["timestamp", "visitorid", "event", "itemid"]
for col in required_event_cols:
    if col not in events.columns:
        events[col] = pd.NA

events["timestamp"] = pd.to_numeric(events["timestamp"], errors="coerce")
events["visitorid"] = pd.to_numeric(events["visitorid"], errors="coerce")
events["itemid"] = pd.to_numeric(events["itemid"], errors="coerce")
events["transactionid"] = pd.to_numeric(events.get("transactionid"), errors="coerce")
events["event"] = events["event"].astype("string").str.strip().str.lower()

events = events[events["event"].isin(EVENT_WEIGHTS.keys())]
events = events.dropna(subset=["timestamp", "visitorid", "itemid"])
events = events.drop_duplicates(subset=["timestamp", "visitorid", "event", "itemid"])

events["event_ts"] = pd.to_datetime(events["timestamp"], unit="ms", errors="coerce")
events = events.dropna(subset=["event_ts"])

events["event_weight"] = events["event"].map(EVENT_WEIGHTS)
events["event_date"] = events["event_ts"].dt.date.astype("string")
events["event_hour"] = events["event_ts"].dt.hour
events["event_dayofweek"] = events["event_ts"].dt.dayofweek
events["event_month"] = events["event_ts"].dt.month

# Normalize timestamp as recency score
max_ts = events["event_ts"].max()
events["days_from_latest"] = (max_ts - events["event_ts"]).dt.total_seconds() / 86400.0
events = minmax_scale_columns(events, ["days_from_latest", "event_hour"])

print("Cleaned events:", events.shape)

# =========================
# 6) Build item-category mapping from Retailrocket item properties
# =========================
item_properties = item_properties.copy()

for col in ["timestamp", "itemid"]:
    item_properties[col] = pd.to_numeric(item_properties[col], errors="coerce")

item_properties["property"] = item_properties["property"].astype("string").str.strip().str.lower()
item_properties["value"] = item_properties["value"].astype("string").str.strip()

item_properties = item_properties.dropna(subset=["timestamp", "itemid", "property", "value"])
item_properties["prop_ts"] = pd.to_datetime(item_properties["timestamp"], unit="ms", errors="coerce")
item_properties = item_properties.dropna(subset=["prop_ts"])

category_candidates = item_properties[item_properties["property"].isin(["categoryid", "category"])].copy()
category_candidates = category_candidates.sort_values(["itemid", "prop_ts"]).drop_duplicates("itemid", keep="last")
category_candidates = category_candidates.rename(columns={"value": "category_value"})
item_category_map = category_candidates[["itemid", "category_value"]].copy()

if "categoryid" in category_tree.columns:
    category_tree["categoryid"] = pd.to_numeric(category_tree["categoryid"], errors="coerce")
if "parentid" in category_tree.columns:
    category_tree["parentid"] = pd.to_numeric(category_tree["parentid"], errors="coerce")

item_category_map["category_value_num"] = pd.to_numeric(item_category_map["category_value"], errors="coerce")
item_category_map = item_category_map.merge(
    category_tree,
    left_on="category_value_num",
    right_on="categoryid",
    how="left"
)

item_category_map["item_category_encoded"] = item_category_map["category_value"].astype("string").fillna("unknown")

# =========================
# 7) Clean DummyJSON products
# =========================
products = products.copy()

expected_product_cols = ["id", "title", "category", "price", "rating", "stock"]
for col in expected_product_cols:
    if col not in products.columns:
        products[col] = pd.NA

for col in ["id", "price", "rating", "stock", "discountPercentage", "minimumOrderQuantity", "weight"]:
    if col in products.columns:
        products[col] = pd.to_numeric(products[col], errors="coerce")

for col in ["title", "category", "brand", "availabilityStatus", "shippingInformation", "returnPolicy"]:
    if col in products.columns:
        products[col] = products[col].astype("string").str.strip()

products = products.dropna(subset=["id", "title", "category", "price"])
products = products.drop_duplicates(subset=["id"])

if "rating" in products.columns:
    products["rating"] = products["rating"].clip(lower=1, upper=5)
if "price" in products.columns:
    products["price"] = products["price"].clip(lower=0)
if "stock" in products.columns:
    products["stock"] = products["stock"].clip(lower=0)

products = minmax_scale_columns(products, ["price", "rating", "stock", "discountPercentage"])

# One-hot encoding for category + brand
product_cat_dummies = pd.get_dummies(products["category"], prefix="prod_cat", dtype=int)
if "brand" in products.columns:
    product_brand_dummies = pd.get_dummies(products["brand"].fillna("unknown"), prefix="brand", dtype=int)
else:
    product_brand_dummies = pd.DataFrame(index=products.index)

products_prepared = pd.concat([products, product_cat_dummies, product_brand_dummies], axis=1)

print("Cleaned products:", products_prepared.shape)

# =========================
# 8) Aggregate user-item interactions
# =========================
interactions = events.groupby(["visitorid", "itemid"], as_index=False).agg(
    interaction_count=("event", "size"),
    total_weight=("event_weight", "sum"),
    last_event_ts=("event_ts", "max"),
    mean_event_hour_norm=("event_hour_norm", "mean"),
    unique_event_types=("event", "nunique"),
)

interactions["implicit_feedback"] = (interactions["total_weight"] > 0).astype(int)
interactions["last_event_recency_days"] = (max_ts - interactions["last_event_ts"]).dt.total_seconds() / 86400.0
interactions = minmax_scale_columns(interactions, ["interaction_count", "total_weight", "last_event_recency_days"])

# Handle missing user-item interactions:
# Create sampled dense matrix for EDA / sparsity analysis only.
top_users = interactions["visitorid"].value_counts().head(TOP_USERS_HEATMAP).index
top_items = interactions["itemid"].value_counts().head(TOP_ITEMS_HEATMAP).index

interaction_sample = interactions[
    interactions["visitorid"].isin(top_users) & interactions["itemid"].isin(top_items)
].copy()

dense_grid = (
    pd.MultiIndex.from_product([sorted(top_users), sorted(top_items)], names=["visitorid", "itemid"])
    .to_frame(index=False)
    .merge(
        interaction_sample[["visitorid", "itemid", "implicit_feedback", "total_weight_norm"]],
        on=["visitorid", "itemid"],
        how="left",
    )
)

dense_grid["implicit_feedback"] = dense_grid["implicit_feedback"].fillna(0).astype(int)
dense_grid["total_weight_norm"] = dense_grid["total_weight_norm"].fillna(0.0)

# =========================
# 9) User-level features
# =========================
user_features = events.groupby("visitorid", as_index=False).agg(
    total_events=("event", "size"),
    unique_items=("itemid", "nunique"),
    avg_event_weight=("event_weight", "mean"),
    last_seen_ts=("event_ts", "max"),
)

user_features["days_since_last_seen"] = (max_ts - user_features["last_seen_ts"]).dt.total_seconds() / 86400.0
user_features = minmax_scale_columns(user_features, ["total_events", "unique_items", "avg_event_weight", "days_since_last_seen"])

user_features["user_segment"] = pd.cut(
    user_features["total_events"],
    bins=[-1, 5, 20, 1000000],
    labels=["low_activity", "mid_activity", "high_activity"]
).astype("string")

user_segment_dummies = pd.get_dummies(user_features["user_segment"], prefix="user_seg", dtype=int)
user_features = pd.concat([user_features, user_segment_dummies], axis=1)

# =========================
# 10) Item-level features from Retailrocket
# =========================
item_features = events.groupby("itemid", as_index=False).agg(
    item_interactions=("event", "size"),
    unique_users=("visitorid", "nunique"),
    mean_weight=("event_weight", "mean"),
    last_item_event_ts=("event_ts", "max"),
)

item_features["days_since_last_item_event"] = (
    max_ts - item_features["last_item_event_ts"]
).dt.total_seconds() / 86400.0

item_features = item_features.merge(
    item_category_map[["itemid", "item_category_encoded", "categoryid", "parentid"]],
    on="itemid",
    how="left"
)

item_features["item_category_encoded"] = item_features["item_category_encoded"].fillna("unknown")
item_cat_dummies = pd.get_dummies(item_features["item_category_encoded"], prefix="rr_cat", dtype=int)
item_features = pd.concat([item_features, item_cat_dummies], axis=1)
item_features = minmax_scale_columns(
    item_features,
    ["item_interactions", "unique_users", "mean_weight", "days_since_last_item_event"]
)

# =========================
# 11) EDA - interaction distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(interactions["interaction_count"], bins=40, kde=True, ax=ax, color="#0076A8")
ax.set_title("Interaction Count Distribution per User-Item Pair")
ax.set_xlabel("interaction_count")
ax.set_ylabel("frequency")
save_plot(fig, "01_interaction_count_histogram.png")

fig, ax = plt.subplots(figsize=(8, 5))
event_counts = events["event"].value_counts().sort_values(ascending=False)
sns.barplot(x=event_counts.index, y=event_counts.values, ax=ax, palette=["#86BC25", "#0076A8", "#1D4F91"])
ax.set_title("Interaction Event Type Distribution")
ax.set_xlabel("event")
ax.set_ylabel("count")
save_plot(fig, "02_event_type_distribution.png")

# =========================
# 12) EDA - item popularity
# =========================
top_popular_items = (
    interactions.groupby("itemid", as_index=False)
    .agg(total_interactions=("interaction_count", "sum"), total_weight=("total_weight", "sum"))
    .sort_values("total_interactions", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=top_popular_items,
    x="total_interactions",
    y=top_popular_items["itemid"].astype(str),
    ax=ax,
    color="#43B02A"
)
ax.set_title("Top 20 Most Popular Items")
ax.set_xlabel("total_interactions")
ax.set_ylabel("itemid")
save_plot(fig, "03_top_item_popularity.png")

# =========================
# 13) EDA - user activity distribution
# =========================
user_activity = interactions.groupby("visitorid", as_index=False).agg(
    items_interacted=("itemid", "nunique"),
    total_interactions=("interaction_count", "sum")
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(user_activity["items_interacted"], bins=40, kde=True, ax=ax, color="#005587")
ax.set_title("User Activity Distribution")
ax.set_xlabel("unique items interacted")
ax.set_ylabel("frequency")
save_plot(fig, "04_user_activity_histogram.png")

# =========================
# 14) EDA - sparsity analysis
# =========================
full_users = interactions["visitorid"].nunique()
full_items = interactions["itemid"].nunique()
observed_pairs = len(interactions)
possible_pairs = full_users * full_items if full_users > 0 and full_items > 0 else 0
sparsity = 1 - (observed_pairs / possible_pairs) if possible_pairs > 0 else np.nan

heatmap_matrix = dense_grid.pivot(index="visitorid", columns="itemid", values="implicit_feedback")

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heatmap_matrix, cmap="YlGnBu", cbar=True, ax=ax)
ax.set_title("User-Item Interaction Sparsity Heatmap")
ax.set_xlabel("itemid")
ax.set_ylabel("visitorid")
save_plot(fig, "05_sparsity_heatmap.png")

# =========================
# 15) Optional EDA - product price distribution
# =========================
if "price" in products_prepared.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(products_prepared["price"], bins=30, kde=True, ax=ax, color="#ED8B00")
    ax.set_title("DummyJSON Product Price Distribution")
    ax.set_xlabel("price")
    ax.set_ylabel("frequency")
    save_plot(fig, "06_product_price_histogram.png")

# =========================
# 16) Build final prepared datasets
# =========================
prepared_interactions = interactions.merge(
    item_features.drop(columns=["last_item_event_ts"], errors="ignore"),
    on="itemid",
    how="left",
    suffixes=("", "_item")
).merge(
    user_features.drop(columns=["last_seen_ts"], errors="ignore"),
    on="visitorid",
    how="left",
    suffixes=("", "_user")
)

prepared_interactions["prep_run_ts"] = RUN_TS
prepared_interactions["source_batch_retailrocket"] = str(retailrocket_batch)
prepared_interactions["source_batch_dummyjson"] = str(dummyjson_batch)

# Sparse matrix artifact for downstream models
user_index = {u: i for i, u in enumerate(sorted(interactions["visitorid"].unique()))}
item_index = {it: i for i, it in enumerate(sorted(interactions["itemid"].unique()))}

rows = interactions["visitorid"].map(user_index)
cols = interactions["itemid"].map(item_index)
vals = interactions["total_weight_norm"].fillna(0).astype(float)

interaction_sparse = csr_matrix((vals, (rows, cols)), shape=(len(user_index), len(item_index)))

# =========================
# 17) Save outputs
# =========================
prepared_interactions_path = PREPARED_DIR / "prepared_interactions.parquet"
user_features_path = PREPARED_DIR / "prepared_user_features.parquet"
item_features_path = PREPARED_DIR / "prepared_item_features.parquet"
products_prepared_path = PREPARED_DIR / "prepared_dummyjson_products.parquet"
dense_grid_path = PREPARED_DIR / "interaction_dense_grid_sample.parquet"
sparse_matrix_path = PREPARED_DIR / "interaction_sparse_matrix.npz"
summary_path = REPORTS_DIR / "preparation_summary.json"
eda_table_path = REPORTS_DIR / "dataset_overview.csv"

prepared_interactions.to_parquet(prepared_interactions_path, index=False)
user_features.to_parquet(user_features_path, index=False)
item_features.to_parquet(item_features_path, index=False)
products_prepared.to_parquet(products_prepared_path, index=False)
dense_grid.to_parquet(dense_grid_path, index=False)
save_npz(sparse_matrix_path, interaction_sparse)

overview_df = pd.DataFrame([
    summarize_df(events, "events_clean"),
    summarize_df(interactions, "user_item_interactions"),
    summarize_df(user_features, "user_features"),
    summarize_df(item_features, "item_features"),
    summarize_df(products_prepared, "dummyjson_products_prepared"),
])
overview_df.to_csv(eda_table_path, index=False)

summary_payload = {
    "run_ts": RUN_TS,
    "retailrocket_batch": str(retailrocket_batch),
    "dummyjson_batch": str(dummyjson_batch),
    "datasets": {
        "events_clean_rows": int(len(events)),
        "interactions_rows": int(len(interactions)),
        "user_features_rows": int(len(user_features)),
        "item_features_rows": int(len(item_features)),
        "dummyjson_products_rows": int(len(products_prepared)),
    },
    "sparsity": {
        "unique_users": int(full_users),
        "unique_items": int(full_items),
        "observed_user_item_pairs": int(observed_pairs),
        "possible_user_item_pairs": int(possible_pairs),
        "sparsity_ratio": float(sparsity) if pd.notna(sparsity) else None,
    },
    "outputs": {
        "prepared_interactions": str(prepared_interactions_path),
        "prepared_user_features": str(user_features_path),
        "prepared_item_features": str(item_features_path),
        "prepared_dummyjson_products": str(products_prepared_path),
        "dense_grid_sample": str(dense_grid_path),
        "sparse_matrix": str(sparse_matrix_path),
        "overview_csv": str(eda_table_path),
    }
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

# =========================
# 18) Display final summary
# =========================
print("\n=== Preparation Summary ===")
print(json.dumps(summary_payload, indent=2))

print("\n=== Prepared Interactions Sample ===")
print(prepared_interactions.head())

print("\n=== User Features Sample ===")
print(user_features.head())

print("\n=== Item Features Sample ===")
print(item_features.head())

print("\n=== DummyJSON Products Sample ===")
print(products_prepared.head())


Found existing installation: matplotlib 3.10.8
Uninstalling matplotlib-3.10.8:
  Successfully uninstalled matplotlib-3.10.8
Found existing installation: seaborn 0.13.2
Uninstalling seaborn-0.13.2:
  Successfully uninstalled seaborn-0.13.2
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.2 MB 4.8 MB/s eta 0:00:02
   -------------- ------------------------- 2.9/8.2 MB 5.6 MB/s eta 0:00:01
   --------------------- ------------------ 4.5/8.2 MB 6.4 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.2 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 7.1 MB/s  0:00:01

   ---------------------------------------- 0/2 [matplotlib]
   ---------------------------------------- 0/2 [matplotlib]
   ---------------------------------------- 0/2 [matplotlib]
   ---------------------------------------- 0/2 [matplotlib]
 

MemoryError: Unable to allocate 17.4 GiB for an array with shape (1087, 2145179) and data type int64